# GraphKnows Relation Extraction Trace: LLM-Free vs DSPy LLM-Assisted

This notebook traces the current GraphKnows extraction paths side by side: the LLM-free dynamic spaCy -> GLiNER2 pass and the optional DSPy `llm_assisted` relation extractor.

The core trace runs the same pipeline class used by ingestion:

```python
GLiNER2EntityExtractor.extract(text)
```

To expose raw logs without reimplementing the pipeline, the notebook wraps only the extractor's dependencies:

- a recording `_SpacySchemaMiner` wrapper records the dynamic schema mined for GLiNER2;
- a recording GLiNER2 model proxy records the exact schema object calls and raw `model.extract(...)` output;
- normalization is taken from the real `GLiNER2EntityExtractor.extract(...)` return value.

So the raw hints and raw GLiNER2 output shown here come from the same module path the STM ingestion pipeline uses. The DSPy section reuses the same input chunk, then shows the RDF ontology inputs, DSPy Predict triplets, audit details, and graph-generation relation metadata.

## Runtime Selectors

These selectors affect only the demo run, optional DSPy relation extraction, and optional topic summarization. Entity labels for the LLM-free path are mined dynamically from the input chunk; DSPy relation labels come only from the configured RDF/OWL ontology file.

In [1]:
# Runtime selectors

TOPIC_SUMMARIZATION_LLM_MODEL = 'deepseek/deepseek-v4-flash'
DSPY_EXTRACTION_LLM_MODEL = 'deepseek/deepseek-v4-flash'
DSPY_ONTOLOGY_FILE = 'graphknows/ingestion/ontology/assets/politics/prov-o.ttl'

# GDS can be slow or unavailable depending on the Neo4j image/plugin setup.
RUN_GDS = True

# Set this to False if you only want the raw extractor trace without touching databases.
RUN_STM_LTM_WRITEBACK = True

# DSPy LLM-assisted relation extraction needs OPENROUTER_API_KEY. If no key is configured,
# the notebook records the exact skip reason instead of failing the LLM-free comparison.
RUN_DSPY_LLM_ASSISTED = True

print({
    'TOPIC_SUMMARIZATION_LLM_MODEL': TOPIC_SUMMARIZATION_LLM_MODEL,
    'DSPY_EXTRACTION_LLM_MODEL': DSPY_EXTRACTION_LLM_MODEL,
    'DSPY_ONTOLOGY_FILE': DSPY_ONTOLOGY_FILE,
    'RUN_GDS': RUN_GDS,
    'RUN_STM_LTM_WRITEBACK': RUN_STM_LTM_WRITEBACK,
    'RUN_DSPY_LLM_ASSISTED': RUN_DSPY_LLM_ASSISTED,
})

{'TOPIC_SUMMARIZATION_LLM_MODEL': 'deepseek/deepseek-v4-flash', 'DSPY_EXTRACTION_LLM_MODEL': 'deepseek/deepseek-v4-flash', 'DSPY_ONTOLOGY_FILE': 'graphknows/ingestion/ontology/assets/politics/prov-o.ttl', 'RUN_GDS': True, 'RUN_STM_LTM_WRITEBACK': True, 'RUN_DSPY_LLM_ASSISTED': True}


## Setup

The imports below are the same GraphKnows modules used in the pipeline. The private imports are intentionally limited to tracing internals already used by `GLiNER2EntityExtractor`; the extraction itself still runs through `GLiNER2EntityExtractor.extract(...)`.

In [2]:
from __future__ import annotations

import importlib
import json
import os
import sys
import time
import uuid
from pathlib import Path
from typing import Any

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'graphknows').exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'graphknows').exists():
    raise RuntimeError('Run this notebook from the graphknows repo root or research/.')
sys.path.insert(0, str(REPO_ROOT))

# Force notebook reruns to use the working-tree graphknows package instead of
# a cached import from an earlier run or installed container image.
importlib.invalidate_caches()
for module_name in [name for name in list(sys.modules) if name == 'graphknows' or name.startswith('graphknows.')]:
    del sys.modules[module_name]

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / '.env')

try:
    from yfiles_jupyter_graphs import GraphWidget
except Exception:
    GraphWidget = None

from graphknows.config import GraphKnowsSettings
from graphknows.config.adapters.gliner_model import load_gliner2_model
from graphknows.config.builders.stores import build_promotion_pipeline, build_stm_service
from graphknows.memory.extractor import GLiNER2EntityExtractor, _BASE_THRESHOLD, _SpacySchemaMiner
from graphknows.memory.models.promotion import PromotionRequest

print('graphknows source root:', REPO_ROOT / 'graphknows')

ONTOLOGY_FILE = str((REPO_ROOT / DSPY_ONTOLOGY_FILE).resolve())
os.environ['GRAPHKNOWS_ONTOLOGY_FILE'] = ONTOLOGY_FILE

base_settings = GraphKnowsSettings()
base_settings.relation_extraction_mode = 'gliner2'
base_settings.summarization_model = TOPIC_SUMMARIZATION_LLM_MODEL

run_id = f'dynamic-trace-{uuid.uuid4().hex[:8]}'
artifact_dir = REPO_ROOT / 'research' / 'notebook_artifacts' / run_id
artifact_dir.mkdir(parents=True, exist_ok=True)

trace: dict[str, Any] = {
    'run_id': run_id,
    'artifact_dir': str(artifact_dir),
    'selectors': {
        'topic_summarization_llm_model': TOPIC_SUMMARIZATION_LLM_MODEL,
        'run_gds': RUN_GDS,
        'run_stm_ltm_writeback': RUN_STM_LTM_WRITEBACK,
        'run_dspy_llm_assisted': RUN_DSPY_LLM_ASSISTED,
        'dspy_ontology_file': ONTOLOGY_FILE,
    },
    'components': {},
}

def show(value: Any) -> None:
    print(json.dumps(value, indent=2, ensure_ascii=False, default=str))

show({'run_id': run_id, 'artifact_dir': str(artifact_dir), 'selectors': trace['selectors']})

graphknows source root: /app/graphknows
{
  "run_id": "dynamic-trace-6cb1e9ff",
  "artifact_dir": "/app/research/notebook_artifacts/dynamic-trace-6cb1e9ff",
  "selectors": {
    "topic_summarization_llm_model": "deepseek/deepseek-v4-flash",
    "run_gds": true,
    "run_stm_ltm_writeback": true,
    "run_dspy_llm_assisted": true,
    "dspy_ontology_file": "/app/graphknows/ingestion/ontology/assets/politics/prov-o.ttl"
  }
}


## Input Chunk

Use a multi-sentence chunk so GLiNER2 can see cross-sentence context while spaCy mines local syntactic hints.

In [3]:
input_text = (
    'European Commission President Ursula von der Leyen said in Brussels that the Commission supports the Green Deal. '
    'The European Parliament criticized parts of the migration pact, while Germany backed the climate package. '
    'Later, von der Leyen met German Chancellor Olaf Scholz, and Scholz praised the Commission plan.'
)

trace['input_text'] = input_text
print(input_text)

European Commission President Ursula von der Leyen said in Brussels that the Commission supports the Green Deal. The European Parliament criticized parts of the migration pact, while Germany backed the climate package. Later, von der Leyen met German Chancellor Olaf Scholz, and Scholz praised the Commission plan.


# 1. Recording Wrappers Around the Real Pipeline Modules

These wrappers do not replace GraphKnows extraction logic. They only record arguments and return values while delegating to the same `_SpacySchemaMiner`, GLiNER2 model, and `GLiNER2EntityExtractor.extract(...)` call used by the pipeline.

In [4]:
class RecordingSchemaMiner:
    def __init__(self, inner: _SpacySchemaMiner) -> None:
        self.inner = inner
        self.calls: list[dict[str, Any]] = []

    def build(self, text: str) -> Any:
        schema = self.inner.build(text)
        self.calls.append({
            'component': '_SpacySchemaMiner.build',
            'input': {'text': text},
            'output': {
                'entity_spec_passed_to_gliner2': schema.entity_spec,
                'relation_spec_passed_to_gliner2': schema.relation_spec,
                'gliner_label_to_graphknows_type': schema.label_to_type,
            },
        })
        return schema


class RecordingSchema:
    def __init__(self, inner: Any, calls: list[dict[str, Any]]) -> None:
        self.inner = inner
        self.calls = calls

    def entities(self, entity_spec: dict[str, Any]) -> Any:
        self.calls.append({'method': 'schema.entities', 'input': entity_spec})
        return self.inner.entities(entity_spec)

    def relations(self, relation_spec: dict[str, str]) -> Any:
        self.calls.append({'method': 'schema.relations', 'input': relation_spec})
        return self.inner.relations(relation_spec)

    def __getattr__(self, name: str) -> Any:
        return getattr(self.inner, name)


class RecordingGLiNER2Model:
    def __init__(self, inner: Any) -> None:
        self.inner = inner
        self.schema_calls: list[dict[str, Any]] = []
        self.extract_calls: list[dict[str, Any]] = []

    def create_schema(self) -> RecordingSchema:
        raw_schema = self.inner.create_schema()
        self.schema_calls.append({'method': 'model.create_schema', 'output_type': type(raw_schema).__name__})
        return RecordingSchema(raw_schema, self.schema_calls)

    def extract(self, text: str, schema: Any, threshold: float, *args: Any, **kwargs: Any) -> Any:
        start = time.monotonic()
        raw_schema = schema.inner if isinstance(schema, RecordingSchema) else schema
        output = self.inner.extract(text, raw_schema, threshold, *args, **kwargs)
        self.extract_calls.append({
            'component': 'GLiNER2.extract',
            'input': {
                'text': text,
                'threshold': threshold,
                'args': args,
                'kwargs': kwargs,
            },
            'output_raw': output,
            'elapsed_s': time.monotonic() - start,
        })
        return output

    def __getattr__(self, name: str) -> Any:
        return getattr(self.inner, name)


recorded_model_holder: dict[str, Any] = {}

def recording_model_loader() -> RecordingGLiNER2Model:
    raw_model = load_gliner2_model()
    recording_model = RecordingGLiNER2Model(raw_model)
    recorded_model_holder['model'] = recording_model
    return recording_model

recording_schema_miner = RecordingSchemaMiner(_SpacySchemaMiner(model_name='en_core_web_lg'))

show({
    'recording': 'ready',
    'extractor_module': 'graphknows.memory.extractor.GLiNER2EntityExtractor',
    'schema_miner_module': 'graphknows.memory.extractor._SpacySchemaMiner',
    'model_loader_module': 'graphknows.config.adapters.gliner_model.load_gliner2_model',
})


{
  "recording": "ready",
  "extractor_module": "graphknows.memory.extractor.GLiNER2EntityExtractor",
  "schema_miner_module": "graphknows.memory.extractor._SpacySchemaMiner",
  "model_loader_module": "graphknows.config.adapters.gliner_model.load_gliner2_model"
}


# 2. Run the Exact Extractor Path Once

This is the important cell. It calls `GLiNER2EntityExtractor.extract(input_text)` exactly as the ingestion pipeline does. The following cells only print what the wrappers recorded from that one call.

In [5]:
extractor = GLiNER2EntityExtractor(
    model_loader=recording_model_loader,
    schema_miner=recording_schema_miner,
)

start = time.monotonic()
extraction_result = extractor.extract(input_text)
extract_elapsed_s = time.monotonic() - start
recording_model = recorded_model_holder['model']

pipeline_trace = {
    'component': 'GLiNER2EntityExtractor.extract',
    'input': {'text': input_text},
    'output': {
        'entities': extraction_result.entities,
        'relations': extraction_result.relations,
    },
    'elapsed_s': extract_elapsed_s,
}

trace['components']['pipeline_extract'] = pipeline_trace
show(pipeline_trace)

2026-06-09 12:52:25.406880440 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card0": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


You are using a model of type extractor to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first


{
  "component": "GLiNER2EntityExtractor.extract",
  "input": {
    "text": "European Commission President Ursula von der Leyen said in Brussels that the Commission supports the Green Deal. The European Parliament criticized parts of the migration pact, while Germany backed the climate package. Later, von der Leyen met German Chancellor Olaf Scholz, and Scholz praised the Commission plan."
  },
  "output": {
    "entities": [
      {
        "name": "European Parliament",
        "type": "ORG",
        "score": 0.8455721139907837
      },
      {
        "name": "Commission",
        "type": "ORG",
        "score": 0.5644286870956421
      },
      {
        "name": "Germany",
        "type": "ORG",
        "score": 0.5405686497688293
      },
      {
        "name": "Ursula von der Leyen",
        "type": "PERSON",
        "score": 0.9980708956718445
      },
      {
        "name": "Olaf Scholz",
        "type": "PERSON",
        "score": 0.9973434805870056
      },
      {
        "

# 3. What spaCy Hinted for GLiNER2

This is the schema mined by `_SpacySchemaMiner.build(...)` during the real extractor call above.

In [6]:
spacy_schema_hint_trace = recording_schema_miner.calls[-1]
trace['components']['spacy_schema_hints'] = spacy_schema_hint_trace
show(spacy_schema_hint_trace)

{
  "component": "_SpacySchemaMiner.build",
  "input": {
    "text": "European Commission President Ursula von der Leyen said in Brussels that the Commission supports the Green Deal. The European Parliament criticized parts of the migration pact, while Germany backed the climate package. Later, von der Leyen met German Chancellor Olaf Scholz, and Scholz praised the Commission plan."
  },
  "output": {
    "entity_spec_passed_to_gliner2": {
      "org": {
        "description": "Named entity with the label ORG, observed in this chunk.",
        "threshold": 0.35
      },
      "person": {
        "description": "Named entity with the label PERSON, observed in this chunk.",
        "threshold": 0.35
      },
      "gpe": {
        "description": "Named entity with the label GPE, observed in this chunk.",
        "threshold": 0.35
      },
      "norp": {
        "description": "Named entity with the label NORP, observed in this chunk.",
        "threshold": 0.35
      }
    },
    "relat

# 4. What Was Passed Into the GLiNER2 Schema Object

These are the actual `schema.entities(...)` and `schema.relations(...)` calls made by `GLiNER2EntityExtractor._build_gliner_schema(...)` during `extract(...)`.

In [7]:
gliner_schema_call_trace = {
    'component': 'GLiNER2 schema construction inside GLiNER2EntityExtractor.extract',
    'calls': recording_model.schema_calls,
}
trace['components']['gliner_schema_calls'] = gliner_schema_call_trace
show(gliner_schema_call_trace)

{
  "component": "GLiNER2 schema construction inside GLiNER2EntityExtractor.extract",
  "calls": [
    {
      "method": "model.create_schema",
      "output_type": "Schema"
    },
    {
      "method": "schema.entities",
      "input": {
        "org": {
          "description": "Named entity with the label ORG, observed in this chunk.",
          "threshold": 0.35
        },
        "person": {
          "description": "Named entity with the label PERSON, observed in this chunk.",
          "threshold": 0.35
        },
        "gpe": {
          "description": "Named entity with the label GPE, observed in this chunk.",
          "threshold": 0.35
        },
        "norp": {
          "description": "Named entity with the label NORP, observed in this chunk.",
          "threshold": 0.35
        }
      }
    },
    {
      "method": "schema.relations",
      "input": {
        "support": "A relation between two concise entity or noun-phrase arguments expressed by the observed verb tr

# 5. Raw GLiNER2 Output

This is the raw dictionary returned by `GLiNER2.extract(...)` during the real extractor call, before GraphKnows normalized it into `ExtractionResult`.

In [8]:
raw_gliner_trace = recording_model.extract_calls[-1]
trace['components']['raw_gliner2_output'] = raw_gliner_trace
show(raw_gliner_trace)

{
  "component": "GLiNER2.extract",
  "input": {
    "text": "European Commission President Ursula von der Leyen said in Brussels that the Commission supports the Green Deal. The European Parliament criticized parts of the migration pact, while Germany backed the climate package. Later, von der Leyen met German Chancellor Olaf Scholz, and Scholz praised the Commission plan.",
    "threshold": 0.35,
    "args": [
      true,
      true,
      true
    ],
    "kwargs": {}
  },
  "output_raw": {
    "entities": {
      "org": [
        {
          "text": "European Parliament",
          "confidence": 0.8455721139907837,
          "start": 117,
          "end": 136
        },
        {
          "text": "Commission",
          "confidence": 0.5644286870956421,
          "start": 9,
          "end": 19
        },
        {
          "text": "Germany",
          "confidence": 0.5405686497688293,
          "start": 183,
          "end": 190
        }
      ],
      "person": [
        {
    

# 6. Compact Raw Debug Log

This cell gathers the dynamic hints, schema calls, raw GLiNER2 output, and normalized GraphKnows output in one object.

In [9]:
debug_log = {
    '1_spacy_hints_from_pipeline_schema_miner': spacy_schema_hint_trace['output'],
    '2_gliner2_schema_calls_from_pipeline_extractor': recording_model.schema_calls,
    '3_raw_gliner2_output': raw_gliner_trace['output_raw'],
    '4_graphknows_normalized_extraction_result': pipeline_trace['output'],
}
trace['components']['debug_log'] = debug_log
show(debug_log)

{
  "1_spacy_hints_from_pipeline_schema_miner": {
    "entity_spec_passed_to_gliner2": {
      "org": {
        "description": "Named entity with the label ORG, observed in this chunk.",
        "threshold": 0.35
      },
      "person": {
        "description": "Named entity with the label PERSON, observed in this chunk.",
        "threshold": 0.35
      },
      "gpe": {
        "description": "Named entity with the label GPE, observed in this chunk.",
        "threshold": 0.35
      },
      "norp": {
        "description": "Named entity with the label NORP, observed in this chunk.",
        "threshold": 0.35
      }
    },
    "relation_spec_passed_to_gliner2": {
      "support": "A relation between two concise entity or noun-phrase arguments expressed by the observed verb trigger 'support' or a close paraphrase. Prefer direct subject, object, complement, or prepositional-object arguments; avoid selecting whole clauses as relation heads or tails. Observed dependency frames in this 

# 7. DSPy LLM-Assisted Relation Extraction Details

This section runs the optional `llm_assisted` relation extractor next to the LLM-free GLiNER2 trace. It uses only the input chunk plus the configured RDF/OWL ontology file; it does not use GLiNER2 entity hints, GLiNER2 candidate relations, taxonomy YAML, or fallback labels. The output is the DSPy triplet audit plus the relation tuples that graph generation writes as metadata-bearing edges.

In [10]:
from graphknows.graphgen.extraction.relations import DSPyExtractor
from graphknows.ingestion.ontology.loader import load_rdf_ontology_file


rdf_ontology = load_rdf_ontology_file(ONTOLOGY_FILE)
ontology_classes = [label.upper().replace(' ', '_').replace('-', '_') for label in rdf_ontology.class_labels]
ontology_relation_types = [label.upper().replace(' ', '_').replace('-', '_') for label in rdf_ontology.property_labels]

dspy_signature_inputs = {
    'text': input_text,
    'ontology_context': rdf_ontology.context,
    'ontology_classes': ontology_classes,
    'ontology_relation_types': ontology_relation_types,
}

dspy_extraction_trace = {
    'enabled': bool(RUN_DSPY_LLM_ASSISTED),
    'dspy_program': 'dspy.Predict(EntityRelation)',
    'signature': {
        'inputs': ['text', 'ontology_context', 'ontology_classes', 'ontology_relation_types'],
        'output': 'triplets: list[Triplet]',
    },
    'active_rdf_ontology': {
        'ontology_file': rdf_ontology.file_path,
        'entity_label_count': len(ontology_classes),
        'relation_label_count': len(ontology_relation_types),
        'relation_labels': ontology_relation_types,
    },
    'signature_inputs': dspy_signature_inputs,
}

if not RUN_DSPY_LLM_ASSISTED:
    dspy_extraction_trace['status'] = 'skipped: RUN_DSPY_LLM_ASSISTED=False'
elif not (base_settings.openrouter_api_key or '').strip():
    dspy_extraction_trace['status'] = 'skipped: OPENROUTER_API_KEY is not configured'
else:
    start = time.monotonic()
    dspy_extractor = DSPyExtractor(
        config={
            'llm': {'model': DSPY_EXTRACTION_LLM_MODEL},
            'infra': {'openrouter_api_key': base_settings.openrouter_api_key or ''},
            'ontology': {'file': ONTOLOGY_FILE},
        }
    )
    dspy_relations, dspy_nodes, dspy_audit = await dspy_extractor.extract_relations(
        text=input_text,
    )
    dspy_extraction_trace.update({
        'status': 'ran',
        'elapsed_s': time.monotonic() - start,
        'relations_for_graph_generation': dspy_relations,
        'nodes_data': dspy_nodes,
        'audit': dspy_audit,
    })

trace['components']['dspy_llm_assisted_relation_extraction'] = dspy_extraction_trace
show(dspy_extraction_trace)

2026/06/09 12:53:16 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=2048. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.0)  if the reason for truncation is repetition.


{
  "enabled": true,
  "dspy_program": "dspy.Predict(EntityRelation)",
  "signature": {
    "inputs": [
      "text",
      "ontology_context",
      "ontology_classes",
      "ontology_relation_types"
    ],
    "output": "triplets: list[Triplet]"
  },
  "active_rdf_ontology": {
    "ontology_file": "/app/graphknows/ingestion/ontology/assets/politics/prov-o.ttl",
    "entity_label_count": 59,
    "relation_label_count": 68,
    "relation_labels": [
      "ACTEDONBEHALFOF",
      "ACTIVITY",
      "AGENT",
      "ALTERNATEOF",
      "ASINBUNDLE",
      "ATLOCATION",
      "ATTIME",
      "DERIVEDBYINSERTIONFROM",
      "DERIVEDBYREMOVALFROM",
      "DESCRIBESSERVICE",
      "DICTIONARY",
      "ENDEDATTIME",
      "ENTITY",
      "GENERATED",
      "GENERATEDATTIME",
      "HADACTIVITY",
      "HADDICTIONARYMEMBER",
      "HADGENERATION",
      "HADMEMBER",
      "HADPLAN",
      "HADPRIMARYSOURCE",
      "HADROLE",
      "HADUSAGE",
      "HASPROVENANCESERVICE",
      "HAS_ANCHOR",
  

# 8. LLM-Free vs DSPy Relation Comparison

This compact object keeps the normalized LLM-free GLiNER2 relations and the DSPy graph-generation relations side by side.

In [11]:
relation_comparison = {
    'llm_free_gliner2_relations': extraction_result.relations,
    'llm_free_gliner2_relation_count': len(extraction_result.relations),
    'dspy_llm_assisted_status': dspy_extraction_trace.get('status'),
    'dspy_llm_assisted_relations_for_graph_generation': dspy_extraction_trace.get('relations_for_graph_generation', []),
    'dspy_llm_assisted_relation_count': len(dspy_extraction_trace.get('relations_for_graph_generation', [])),
}
trace['components']['relation_comparison'] = relation_comparison
show(relation_comparison)

{
  "llm_free_gliner2_relations": [
    {
      "head": "Commission",
      "head_type": "ORG",
      "relation": "support",
      "tail": "Green Deal",
      "tail_type": "ENTITY",
      "score": 0.9685333967208862
    },
    {
      "head": "von der Leyen",
      "head_type": "ENTITY",
      "relation": "plan",
      "tail": "Commission plan",
      "tail_type": "ENTITY",
      "score": 0.5232700705528259
    },
    {
      "head": "Ursula von der Leyen",
      "head_type": "PERSON",
      "relation": "say",
      "tail": "Commission supports the Green Deal",
      "tail_type": "ENTITY",
      "score": 0.43024617433547974
    },
    {
      "head": "Scholz",
      "head_type": "ENTITY",
      "relation": "praise",
      "tail": "Commission plan",
      "tail_type": "ENTITY",
      "score": 0.9947866201400757
    },
    {
      "head": "Ursula von der Leyen",
      "head_type": "PERSON",
      "relation": "meet",
      "tail": "Olaf Scholz",
      "tail_type": "PERSON",
      "score":

# 9. New Benchmark Evaluation Comparison

This section compares completed evaluation artifacts from the new memory evaluation framework: LoCoMo, LongMemEval, and BEAM. It compares LLM-free `gliner2` relation extraction against LLM-assisted `llm_assisted` relation extraction.

It deliberately does not run benchmarks; it only loads existing JSON reports from `evaluation/results` or from explicit paths configured below. Generate matching report pairs with the evaluation CLI by running the same benchmark and case slice twice with `--promote-to-ltm --recall-scope ltm`, changing only `--relation-extraction-mode` between `gliner2` and `llm_assisted`. Promotion is required because LLM-assisted relation extraction happens in the STM -> LTM pipeline, not in plain STM recall.

The CLI also supports `off` for `--relation-extraction-mode`; this notebook treats it as an optional no-relation ablation when a matching report is available.

In [12]:
# Optional explicit report paths. Leave values empty to auto-discover the newest matching reports.
# These cells load artifacts only; they do not run evaluation commands.
SUPPORTED_EVALUATION_BENCHMARKS = ('locomo', 'longmemeval', 'beam')

BENCHMARK_REPORT_PAIRS = {
    'locomo': {
        'llm_free': '',
        'llm_assisted': '',
        'relation_off': '',
    },
    'longmemeval': {
        'llm_free': '',
        'llm_assisted': '',
        'relation_off': '',
    },
    'beam': {
        'llm_free': '',
        'llm_assisted': '',
        'relation_off': '',
    },
}

BENCHMARK_MODE_LABELS = {
    'llm_free': 'gliner2',
    'llm_assisted': 'llm_assisted',
    'relation_off': 'off',
}

BENCHMARK_RUN_COMMANDS = {
    benchmark: {
        bucket: (
            f'python -m evaluation {benchmark} '
            f'--relation-extraction-mode {mode} '
            f'--promote-to-ltm --recall-scope ltm '
            f'--run-id eval-{benchmark}-{mode}'
        )
        for bucket, mode in BENCHMARK_MODE_LABELS.items()
    }
    for benchmark in SUPPORTED_EVALUATION_BENCHMARKS
}

BENCHMARK_RESULT_ROOT = REPO_ROOT / 'evaluation' / 'results'

show({
    'supported_evaluation_benchmarks': SUPPORTED_EVALUATION_BENCHMARKS,
    'result_root': str(BENCHMARK_RESULT_ROOT),
    'configured_pairs': BENCHMARK_REPORT_PAIRS,
    'mode_labels': BENCHMARK_MODE_LABELS,
    'run_command_templates': BENCHMARK_RUN_COMMANDS,
})

{
  "supported_evaluation_benchmarks": [
    "locomo",
    "longmemeval",
    "beam"
  ],
  "result_root": "/app/evaluation/results",
  "configured_pairs": {
    "locomo": {
      "llm_free": "",
      "llm_assisted": "",
      "relation_off": ""
    },
    "longmemeval": {
      "llm_free": "",
      "llm_assisted": "",
      "relation_off": ""
    },
    "beam": {
      "llm_free": "",
      "llm_assisted": "",
      "relation_off": ""
    }
  },
  "mode_labels": {
    "llm_free": "gliner2",
    "llm_assisted": "llm_assisted",
    "relation_off": "off"
  },
  "run_command_templates": {
    "locomo": {
      "llm_free": "python -m evaluation locomo --relation-extraction-mode gliner2 --promote-to-ltm --recall-scope ltm --run-id eval-locomo-gliner2",
      "llm_assisted": "python -m evaluation locomo --relation-extraction-mode llm_assisted --promote-to-ltm --recall-scope ltm --run-id eval-locomo-llm_assisted",
      "relation_off": "python -m evaluation locomo --relation-extraction-mode

In [13]:
def _load_benchmark_report(path: Path) -> dict[str, Any]:
    report = json.loads(path.read_text(encoding='utf-8'))
    required = {'benchmark', 'run_id', 'summary', 'cases'}
    missing = sorted(required - set(report))
    if missing:
        raise ValueError(f'{path} is not a benchmark report; missing {missing}')
    if not isinstance(report.get('cases'), list):
        raise ValueError(f'{path} has non-list cases')
    return report


def _infer_relation_mode(report: dict[str, Any], path: Path) -> str | None:
    metadata = report.get('metadata', {}) if isinstance(report.get('metadata'), dict) else {}
    direct = metadata.get('relation_extraction_mode') or metadata.get('relation_mode')
    if direct in {'gliner2', 'llm_assisted', 'off'}:
        return str(direct)
    haystack = f"{report.get('run_id', '')} {path.name} {path.parent.name}".lower()
    if 'llm_assisted' in haystack or 'llm-assisted' in haystack:
        return 'llm_assisted'
    if 'gliner2' in haystack or 'llm_free' in haystack or 'llm-free' in haystack:
        return 'gliner2'
    if '-off-' in haystack or haystack.endswith('-off') or 'relation-off' in haystack:
        return 'off'
    return None


def _mode_bucket(mode: str | None) -> str | None:
    return {
        'gliner2': 'llm_free',
        'llm_assisted': 'llm_assisted',
        'off': 'relation_off',
    }.get(mode or '')


def _canonical_benchmark_name(report: dict[str, Any]) -> str:
    return str(report.get('benchmark', '')).lower()


def _discover_benchmark_reports(result_root: Path) -> dict[str, dict[str, dict[str, Any]]]:
    discovered: dict[str, dict[str, dict[str, Any]]] = {}
    for path in sorted(result_root.glob('**/*.json'), key=lambda item: item.stat().st_mtime, reverse=True):
        try:
            report = _load_benchmark_report(path)
        except Exception:
            continue
        benchmark = _canonical_benchmark_name(report)
        if benchmark not in SUPPORTED_EVALUATION_BENCHMARKS:
            continue
        metadata = report.get('metadata', {}) if isinstance(report.get('metadata'), dict) else {}
        if metadata.get('promote_to_ltm') is not True:
            continue
        bucket = _mode_bucket(_infer_relation_mode(report, path))
        if bucket is None:
            continue
        discovered.setdefault(benchmark, {}).setdefault(bucket, {'path': path, 'report': report})
    return discovered


def _resolve_configured_reports() -> dict[str, dict[str, dict[str, Any]]]:
    resolved = _discover_benchmark_reports(BENCHMARK_RESULT_ROOT)
    for benchmark, modes in BENCHMARK_REPORT_PAIRS.items():
        if benchmark not in SUPPORTED_EVALUATION_BENCHMARKS:
            continue
        for bucket, raw_path in modes.items():
            if not raw_path:
                continue
            path = Path(raw_path)
            if not path.is_absolute():
                path = REPO_ROOT / path
            report = _load_benchmark_report(path)
            report_benchmark = _canonical_benchmark_name(report)
            if report_benchmark != benchmark:
                raise ValueError(f'{path} is a {report_benchmark!r} report, not {benchmark!r}')
            metadata = report.get('metadata', {}) if isinstance(report.get('metadata'), dict) else {}
            if metadata.get('promote_to_ltm') is not True:
                raise ValueError(f'{path} was not run with --promote-to-ltm')
            resolved.setdefault(benchmark, {})[bucket] = {'path': path, 'report': report}
    return resolved


benchmark_reports_by_mode = _resolve_configured_reports()
show({
    benchmark: {bucket: str(payload['path']) for bucket, payload in modes.items()}
    for benchmark, modes in benchmark_reports_by_mode.items()
})

{
  "beam": {
    "llm_assisted": "/app/evaluation/results/beam/demo-beam-llm_assisted-beam.json",
    "llm_free": "/app/evaluation/results/beam/demo-beam-gliner2-beam.json"
  },
  "longmemeval": {
    "llm_assisted": "/app/evaluation/results/longmemeval/demo-longmemeval-llm_assisted-longmemeval.json",
    "llm_free": "/app/evaluation/results/longmemeval/demo-longmemeval-gliner2-longmemeval.json"
  },
  "locomo": {
    "llm_assisted": "/app/evaluation/results/locomo/demo-locomo-llm_assisted-locomo.json",
    "llm_free": "/app/evaluation/results/locomo/demo-locomo-gliner2-locomo.json"
  }
}


In [14]:
def _numeric_metrics(report: dict[str, Any]) -> dict[str, float]:
    summary = report.get('summary', {}) if isinstance(report.get('summary'), dict) else {}
    metrics: dict[str, float] = {}
    avg_score = summary.get('avg_score')
    if isinstance(avg_score, (int, float)):
        metrics['avg_score'] = float(avg_score)
    metric_summary = summary.get('metric_summary', {})
    if isinstance(metric_summary, dict):
        for key, value in metric_summary.items():
            if isinstance(value, bool):
                metrics[key] = 1.0 if value else 0.0
            elif isinstance(value, (int, float)):
                metrics[key] = float(value)
    return metrics


def _case_ids(report: dict[str, Any]) -> set[str]:
    return {str(case.get('case_id')) for case in report.get('cases', []) if isinstance(case, dict)}


def _compare_report_pair(benchmark: str, modes: dict[str, dict[str, Any]]) -> dict[str, Any]:
    checks = []
    rows = []
    free_payload = modes.get('llm_free')
    assisted_payload = modes.get('llm_assisted')
    if not free_payload:
        checks.append('missing llm_free/gliner2 report')
    if not assisted_payload:
        checks.append('missing llm_assisted report')
    if not free_payload or not assisted_payload:
        return {'benchmark': benchmark, 'status': 'incomplete', 'checks': checks, 'metric_rows': rows}

    free_report = free_payload['report']
    assisted_report = assisted_payload['report']
    if free_report.get('benchmark') != assisted_report.get('benchmark'):
        checks.append(f"benchmark mismatch: {free_report.get('benchmark')} vs {assisted_report.get('benchmark')}")

    free_cases = _case_ids(free_report)
    assisted_cases = _case_ids(assisted_report)
    overlap = free_cases & assisted_cases
    if free_cases != assisted_cases:
        checks.append(
            f'case-id mismatch: overlap={len(overlap)}, llm_free_only={len(free_cases - assisted_cases)}, llm_assisted_only={len(assisted_cases - free_cases)}'
        )

    free_metrics = _numeric_metrics(free_report)
    assisted_metrics = _numeric_metrics(assisted_report)
    for metric in sorted(set(free_metrics) | set(assisted_metrics)):
        free_value = free_metrics.get(metric)
        assisted_value = assisted_metrics.get(metric)
        rows.append({
            'benchmark': benchmark,
            'metric': metric,
            'llm_free_gliner2': free_value,
            'llm_assisted': assisted_value,
            'delta_assisted_minus_free': (
                assisted_value - free_value
                if free_value is not None and assisted_value is not None
                else None
            ),
        })

    off_payload = modes.get('relation_off')
    if off_payload:
        off_metrics = _numeric_metrics(off_payload['report'])
        for metric in sorted(off_metrics):
            rows.append({
                'benchmark': benchmark,
                'metric': f'{metric} (off baseline)',
                'llm_free_gliner2': free_metrics.get(metric),
                'llm_assisted': assisted_metrics.get(metric),
                'relation_off': off_metrics[metric],
            })

    return {
        'benchmark': benchmark,
        'status': 'ok' if not checks else 'check',
        'checks': checks,
        'case_overlap': len(overlap),
        'llm_free_path': str(free_payload['path']),
        'llm_assisted_path': str(assisted_payload['path']),
        'metric_rows': rows,
    }


benchmark_relation_mode_comparison = {
    benchmark: _compare_report_pair(benchmark, benchmark_reports_by_mode.get(benchmark, {}))
    for benchmark in SUPPORTED_EVALUATION_BENCHMARKS
}
trace['components']['benchmark_relation_extraction_comparison'] = benchmark_relation_mode_comparison
show(benchmark_relation_mode_comparison)

{
  "locomo": {
    "benchmark": "locomo",
    "status": "ok",
    "checks": [],
    "case_overlap": 5,
    "llm_free_path": "/app/evaluation/results/locomo/demo-locomo-gliner2-locomo.json",
    "llm_assisted_path": "/app/evaluation/results/locomo/demo-locomo-llm_assisted-locomo.json",
    "metric_rows": [
      {
        "benchmark": "locomo",
        "metric": "answer_support_hit",
        "llm_free_gliner2": 1.0,
        "llm_assisted": 1.0,
        "delta_assisted_minus_free": 0.0
      },
      {
        "benchmark": "locomo",
        "metric": "answer_support_score",
        "llm_free_gliner2": 1.0,
        "llm_assisted": 1.0,
        "delta_assisted_minus_free": 0.0
      },
      {
        "benchmark": "locomo",
        "metric": "avg_score",
        "llm_free_gliner2": 1.0,
        "llm_assisted": 1.0,
        "delta_assisted_minus_free": 0.0
      },
      {
        "benchmark": "locomo",
        "metric": "flush_chunks",
        "llm_free_gliner2": 1.8,
        "llm_assiste

In [15]:
_RECALL_METRICS = {'avg_score', 'answer_support_hit', 'answer_support_score'}
_GRAPH_METRICS = {'flush_edges', 'flush_nodes', 'flush_chunks', 'flush_topics', 'flush_error_count'}


def _fmt_metric(value):
    if value is None:
        return '-'
    if isinstance(value, float):
        return f'{value:.3f}'
    return str(value)


def _fmt_delta(value):
    if value is None:
        return '-'
    if isinstance(value, float):
        sign = '+' if value > 0 else ''
        return f'{sign}{value:.3f}'
    return str(value)


def render_benchmark_relation_mode_markdown(comparison):
    lines = [
        '## Benchmark Evaluation: LLM-free (gliner2) vs LLM-assisted',
        '',
        '> Results averaged over 5 questions per benchmark (smoke fixtures). Both modes use --promote-to-ltm --recall-scope ltm.',
        '',
        '### Recall Quality',
        '',
        '| Benchmark | Metric | gliner2 | llm_assisted | Delta |',
        '| --- | --- | ---: | ---: | ---: |',
    ]
    for benchmark in SUPPORTED_EVALUATION_BENCHMARKS:
        payload = comparison.get(benchmark, {})
        for row in payload.get('metric_rows', []):
            metric = row['metric']
            if metric in _RECALL_METRICS:
                lines.append(
                    f"| {benchmark} | {metric} | {_fmt_metric(row.get('llm_free_gliner2'))} "
                    f"| {_fmt_metric(row.get('llm_assisted'))} "
                    f"| {_fmt_delta(row.get('delta_assisted_minus_free'))} |"
                )
    lines += [
        '',
        '### Graph Construction (STM to LTM flush)',
        '',
        '| Benchmark | Metric | gliner2 | llm_assisted | Delta |',
        '| --- | --- | ---: | ---: | ---: |',
    ]
    for benchmark in SUPPORTED_EVALUATION_BENCHMARKS:
        payload = comparison.get(benchmark, {})
        for row in payload.get('metric_rows', []):
            metric = row['metric']
            if metric in _GRAPH_METRICS:
                lines.append(
                    f"| {benchmark} | {metric} | {_fmt_metric(row.get('llm_free_gliner2'))} "
                    f"| {_fmt_metric(row.get('llm_assisted'))} "
                    f"| {_fmt_delta(row.get('delta_assisted_minus_free'))} |"
                )
    any_rows = any(
        row['metric'] in _RECALL_METRICS or row['metric'] in _GRAPH_METRICS
        for bm, payload in comparison.items()
        for row in payload.get('metric_rows', [])
    )
    if not any_rows:
        lines.append('| - | No complete gliner2/llm_assisted report pairs found. | - | - | - |')
    issues = [
        f"- `{benchmark}`: {'; '.join(payload.get('checks', []))}"
        for benchmark, payload in comparison.items()
        if payload.get('checks')
    ]
    if issues:
        lines += ['', '### Configuration Checks', '', *issues]
    return '\n'.join(lines)


benchmark_relation_mode_markdown = render_benchmark_relation_mode_markdown(benchmark_relation_mode_comparison)
try:
    from IPython.display import Markdown, display
    display(Markdown(benchmark_relation_mode_markdown))
except Exception:
    print(benchmark_relation_mode_markdown)


## Benchmark Evaluation: LLM-free (gliner2) vs LLM-assisted

> Results averaged over 5 questions per benchmark (smoke fixtures). Both modes use --promote-to-ltm --recall-scope ltm.

### Recall Quality

| Benchmark | Metric | gliner2 | llm_assisted | Delta |
| --- | --- | ---: | ---: | ---: |
| locomo | answer_support_hit | 1.000 | 1.000 | 0.000 |
| locomo | answer_support_score | 1.000 | 1.000 | 0.000 |
| locomo | avg_score | 1.000 | 1.000 | 0.000 |
| longmemeval | answer_support_hit | 1.000 | 1.000 | 0.000 |
| longmemeval | answer_support_score | 1.000 | 1.000 | 0.000 |
| longmemeval | avg_score | 1.000 | 1.000 | 0.000 |
| beam | answer_support_hit | 1.000 | 1.000 | 0.000 |
| beam | answer_support_score | 1.000 | 1.000 | 0.000 |
| beam | avg_score | 1.000 | 1.000 | 0.000 |

### Graph Construction (STM to LTM flush)

| Benchmark | Metric | gliner2 | llm_assisted | Delta |
| --- | --- | ---: | ---: | ---: |
| locomo | flush_chunks | 1.800 | 1.800 | 0.000 |
| locomo | flush_edges | 7.800 | 12.200 | +4.400 |
| locomo | flush_error_count | 0.000 | 0.000 | 0.000 |
| locomo | flush_nodes | 7.200 | 7.200 | 0.000 |
| locomo | flush_topics | 0.000 | 0.000 | 0.000 |
| longmemeval | flush_chunks | 1.400 | 1.400 | 0.000 |
| longmemeval | flush_edges | 6.400 | 9.000 | +2.600 |
| longmemeval | flush_error_count | 0.000 | 0.200 | +0.200 |
| longmemeval | flush_nodes | 6.200 | 6.200 | 0.000 |
| longmemeval | flush_topics | 0.000 | 0.000 | 0.000 |
| beam | flush_chunks | 2.000 | 2.000 | 0.000 |
| beam | flush_edges | 5.600 | 10.400 | +4.800 |
| beam | flush_error_count | 0.000 | 0.200 | +0.200 |
| beam | flush_nodes | 6.400 | 6.400 | 0.000 |
| beam | flush_topics | 0.000 | 0.000 | 0.000 |

# 10. Optional STM -> LTM Writeback, GDS Community, Topic Trace

This section runs the normal GraphKnows ingestion and flush path using the dynamic extractor. It touches Memgraph, pgvector, Neo4j, GDS community detection, and DSPy topic summarization.

After the flush, the notebook reads back the actual `TOPIC` nodes and the yielded entity communities from Neo4j so the demo shows topic details, modularity, algorithm metadata, summaries, member entities, and sample chunks.


In [16]:
async def fetch_topic_dissection(flusher: Any) -> dict[str, Any]:
    topic_rows = await flusher._neo._arun(
        """
        MATCH (t:TOPIC)
        OPTIONAL MATCH (e:ENTITY)-[:IN_TOPIC|IN_COMMUNITY]->(t)
        WITH t, collect(DISTINCT e.name) AS members
        OPTIONAL MATCH (c:CHUNK)-[:HAS_ENTITY]->(ce:ENTITY)-[:IN_TOPIC|IN_COMMUNITY]->(t)
        WITH t, members,
             collect(DISTINCT CASE WHEN c IS NULL THEN NULL ELSE {
                 chunk_id: c.id,
                 heading_path: c.heading_path,
                 text: substring(c.text, 0, 360)
             } END) AS chunks
        RETURN t.id AS topic_id,
               t.label AS label,
               t.summary AS summary,
               t.community_id AS community_id,
               t.community_algorithm AS community_algorithm,
               t.modularity AS modularity,
               t.gds_modularity AS gds_modularity,
               t.silhouette AS silhouette,
               t.community_hash AS community_hash,
               t.entity_count AS stored_entity_count,
               size(members) AS linked_entity_count,
               members AS entity_names,
               [chunk IN chunks WHERE chunk IS NOT NULL][..5] AS sample_chunks
        ORDER BY linked_entity_count DESC, topic_id
        """
    )
    community_rows = await flusher._neo._arun(
        """
        MATCH (e:ENTITY)
        WHERE e.community_id IS NOT NULL
        RETURN e.community_id AS community_id,
               collect(DISTINCT e.name) AS entity_names,
               count(DISTINCT e) AS entity_count
        ORDER BY entity_count DESC, community_id
        """
    )
    return {
        'topic_nodes': [dict(row) for row in topic_rows],
        'yielded_entity_communities': [dict(row) for row in community_rows],
    }


async def run_stm_ltm_writeback() -> dict[str, Any]:
    session_id = f'{run_id}-gliner2-dynamic'
    settings = base_settings
    settings.relation_extraction_mode = 'gliner2'
    settings.summarization_model = TOPIC_SUMMARIZATION_LLM_MODEL

    stm_service = build_stm_service(settings)
    ingest_result = stm_service.ingest(
        text=input_text,
        session_id=session_id,
        title='Dynamic spaCy hinted GLiNER2 trace',
        source_hint='research/notebook',
        metadata={'source_hint': 'research/notebook', 'variant': 'dynamic_gliner2'},
    )
    stm_data = await stm_service._mg.read_session_subgraph(session_id)

    flusher = build_promotion_pipeline(settings)
    flusher._llm_model = TOPIC_SUMMARIZATION_LLM_MODEL
    flush_result = await flusher.promote(
        PromotionRequest(
            session_id=session_id,
            clear_stm=False,
            run_gds=RUN_GDS,
            run_topics=True,
            merge_entities=True,
        )
    )
    topics = flusher._neo.list_topic_nodes(min_entities=2)
    topic_dissection = await fetch_topic_dissection(flusher)

    return {
        'session_id': session_id,
        'ingest_result': ingest_result.model_dump(),
        'stm_counts': {key: len(value) for key, value in stm_data.items() if isinstance(value, list)},
        'stm_data': stm_data,
        'flush_result': flush_result.model_dump(),
        'topics': topics,
        'topic_dissection': topic_dissection,
        '_flusher': flusher,
    }

if RUN_STM_LTM_WRITEBACK:
    writeback_trace = await run_stm_ltm_writeback()
    writeback_flusher = writeback_trace.pop('_flusher')
    trace['components']['stm_ltm_writeback'] = writeback_trace
    show({
        'flush_result': writeback_trace['flush_result'],
        'topics': writeback_trace['topics'],
        'topic_dissection': writeback_trace['topic_dissection'],
    })
else:
    writeback_trace = None
    writeback_flusher = None
    print('RUN_STM_LTM_WRITEBACK=False; skipping database writeback trace.')

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `IN_COMMUNITY` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=46, offset=70>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 70, 'line': 3, 'column': 46}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (t:TOPIC)\n        OPTIONAL MATCH (e:ENTITY)-[:IN_TOPIC|IN_COMMUNITY]->(t)\n        WITH t, collect(DISTINCT e.name) AS members\n        OPTIONAL MATCH (c:CHUNK)-[:HAS_ENTITY]->(ce:ENTITY)-[:IN_TOPIC|IN_COMMUNITY]->(t)\n        WITH t, members,\n             collect(DISTINCT CASE WHEN c IS NULL THEN

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `IN_COMMUNITY` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=5, column=72, offset=212>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 212, 'line': 5, 'column': 72}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (t:TOPIC)\n        OPTIONAL MATCH (e:ENTITY)-[:IN_TOPIC|IN_COMMUNITY]->(t)\n        WITH t, collect(DISTINCT e.name) AS members\n        OPTIONAL MATCH (c:CHUNK)-[:HAS_ENTITY]->(ce:ENTITY)-[:IN_TOPIC|IN_COMMUNITY]->(t)\n        WITH t, members,\n             collect(DISTINCT CASE WHEN c IS NULL TH

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `entity_count` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=21, column=18, offset=898>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 898, 'line': 21, 'column': 18}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (t:TOPIC)\n        OPTIONAL MATCH (e:ENTITY)-[:IN_TOPIC|IN_COMMUNITY]->(t)\n        WITH t, collect(DISTINCT e.name) AS members\n        OPTIONAL MATCH (c:CHUNK)-[:HAS_ENTITY]->(ce:ENTITY)-[:IN_TOPIC|IN_COMMUNITY]->(t)\n        WITH t, members,\n             collect(DISTINCT CASE WHEN c IS NULL THEN NULL ELSE

{
  "flush_result": {
    "session_id": "dynamic-trace-6cb1e9ff-gliner2-dynamic",
    "nodes_written": 10,
    "edges_written": 10,
    "chunks_promoted": 0,
    "turns_promoted": 0,
    "topics_created": 0,
    "merged_entities": 7,
    "elapsed_s": 0.2777088191360235,
    "errors": [],
    "gds_result": {
      "entity_count": 107,
      "edge_count": 113,
      "community_count": 0,
      "modularity": 0.0,
      "modularities": [],
      "ran_iterations": 0,
      "did_converge": false,
      "changed_community_ids": [],
      "algorithm_used": "",
      "silhouette_cosine": null,
      "global_separation": null,
      "global_overlap": null,
      "unweighted_modularity": null,
      "error": "gds_unavailable"
    }
  },
  "topics": [
    {
      "topic_id": "5ca65323-611d-546f-94c7-d13b77a116fa",
      "label": "EU Green Deal and Migration Pact: Scholz and von der Leyen Cooperation",
      "community_id": 0,
      "summary": "European Commission President Ursula von der Leyen exp

## Optional yFiles View of LTM Output

This reads back what GraphKnows wrote to Neo4j and formats it for yFiles. It does not run extraction logic.

In [17]:
async def build_yfiles_payload(flusher: Any, title: str) -> dict[str, Any]:
    entity_rows = await flusher._neo._arun(
        "MATCH (e:ENTITY) RETURN e.name AS id, e.name AS label, e.type AS type, e.label AS label_type LIMIT 100"
    )
    relation_rows = await flusher._neo._arun(
        "MATCH (a:ENTITY)-[r]->(b:ENTITY) WHERE type(r) <> 'IN_TOPIC' "
        "RETURN a.name AS source, b.name AS target, type(r) AS label, properties(r) AS props LIMIT 200"
    )
    topic_rows = await flusher._neo._arun(
        "MATCH (t:TOPIC) RETURN t.id AS id, t.label AS label, t.summary AS summary, "
        "t.community_id AS community_id, t.community_algorithm AS community_algorithm, "
        "t.modularity AS modularity, t.gds_modularity AS gds_modularity LIMIT 50"
    )
    topic_edge_rows = await flusher._neo._arun(
        "MATCH (e:ENTITY)-[:IN_TOPIC]->(t:TOPIC) RETURN e.name AS source, t.id AS target, 'IN_TOPIC' AS label LIMIT 200"
    )

    nodes = []
    for raw in entity_rows:
        row = dict(raw)
        nodes.append({'id': row['id'], 'label': row['label'], 'type': row.get('type') or row.get('label_type') or 'ENTITY'})
    for raw in topic_rows:
        row = dict(raw)
        nodes.append({'id': row['id'], 'label': row['label'], 'type': 'TOPIC', 'tag': {'summary': row.get('summary'), 'community_id': row.get('community_id'), 'community_algorithm': row.get('community_algorithm'), 'modularity': row.get('modularity'), 'gds_modularity': row.get('gds_modularity')}})

    edges = []
    for idx, raw in enumerate(relation_rows):
        row = dict(raw)
        edges.append({'id': f'{title}-rel-{idx}', 'start': row['source'], 'end': row['target'], 'source': row['source'], 'target': row['target'], 'label': row['label'], 'tag': row.get('props') or {}})
    for idx, raw in enumerate(topic_edge_rows):
        row = dict(raw)
        edges.append({'id': f'{title}-topic-{idx}', 'start': row['source'], 'end': row['target'], 'source': row['source'], 'target': row['target'], 'label': row['label']})
    return {'nodes': nodes, 'edges': edges}

if writeback_flusher is not None:
    graph_payload = await build_yfiles_payload(writeback_flusher, 'dynamic_gliner2')
    trace['components']['graph_payload'] = graph_payload
    if GraphWidget is None:
        show(graph_payload)
    else:
        widget = GraphWidget()
        widget.nodes = graph_payload['nodes']
        widget.edges = graph_payload['edges']
        display(widget)
else:
    graph_payload = None
    print('No writeback flusher available; skipped graph view.')

GraphWidget(layout=Layout(height='800px', width='100%'))

## Save Trace

The saved JSON contains the raw dynamic schema hints, actual GLiNER2 schema calls, raw GLiNER2 output, normalized LLM-free output, DSPy LLM-assisted signature inputs/audit/comparison, benchmark artifact comparisons when that section is run, and optional STM/LTM traces.

In [18]:
trace_path = artifact_dir / 'relation_extraction_llm_free_vs_dspy_trace.json'
graph_payload_path = artifact_dir / 'dynamic_gliner2_yfiles_graph_payload.json'

trace_path.write_text(json.dumps(trace, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
if graph_payload is not None:
    graph_payload_path.write_text(json.dumps(graph_payload, indent=2, ensure_ascii=False, default=str), encoding='utf-8')

show({
    'trace_path': str(trace_path),
    'graph_payload_path': str(graph_payload_path) if graph_payload is not None else None,
})

{
  "trace_path": "/app/research/notebook_artifacts/dynamic-trace-6cb1e9ff/relation_extraction_llm_free_vs_dspy_trace.json",
  "graph_payload_path": "/app/research/notebook_artifacts/dynamic-trace-6cb1e9ff/dynamic_gliner2_yfiles_graph_payload.json"
}
